# 07. SVD-Based Task Interference Analysis (IF vs Math)

RL fine-tuning으로 생성된 **IF(Instruction Following)**와 **Math** 두 task의 모델을 병합할 때 발생하는
**파괴적 간섭(Destructive Interference)**을 **기하학적 관점(Geometric Perspective)**에서 분석한다.

---

## Mathematical Background

### 1. Task Vector Decomposition
각 layer의 2D weight matrix $W$에 대해 task vector를 정의한다:
- $\Delta_{\text{if}} = W_{\text{if}} - W_{\text{base}}$
- $\Delta_{\text{math}} = W_{\text{math}} - W_{\text{base}}$

SVD로 분해: $\Delta_i = U_i \Sigma_i V_i^\top$

### 2. Singular Task Interference (STI)
$$\text{STI}(\{\Delta_i\}_{i=1}^{T}) = \| (U^\top U - I) \Sigma (V^\top V - I) \|_1$$

- $U = [U_1 | U_2]$: left singular vectors의 수평 결합
- $V = [V_1 | V_2]$: right singular vectors의 수평 결합
- $\Sigma = \text{block\_diag}(\Sigma_1, \Sigma_2)$: singular values의 block diagonal
- **STI = 0** → task subspace들이 완벽히 직교 (간섭 없음)
- **STI >> 0** → subspace overlap으로 인한 파괴적 간섭 존재

### 3. Key Analyses
- **A. Rank Analysis**: Singular value spectrum의 감소 속도 → low-rank 여부
- **B. Cosine Similarity**: $U_{\text{if}}^\top U_{\text{math}}$, $V_{\text{if}}^\top V_{\text{math}}$ → 간섭 위치
- **C. STI per Layer**: Layer depth에 따른 간섭 정량화
- **D. Cumulative Energy**: 상위 k개 singular value로 90/95/99% 재구성 가능한지

In [ ]:
"""Cell 1: Imports and Configuration.

모든 라이브러리를 로드하고 모델 경로, 분석 파라미터, 출력 경로를 설정한다.
safetensors를 직접 사용하여 AutoModelForCausalLM 대비 ~12GB 메모리를 절약한다.
"""
from __future__ import annotations

import gc
import json
import re
from collections import defaultdict
from dataclasses import dataclass, field, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from safetensors.torch import load_file as safetensors_load_file
from tqdm.auto import tqdm

# ──────────────────────────────────────────────────────────────
# Model paths
# ──────────────────────────────────────────────────────────────
BASE_MODEL_ID = "Qwen/Qwen3-1.7B"  # HuggingFace hub ID (transformers cache에서 resolve)

IF_MODEL_PATH = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/"
    "Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface"
)
MATH_MODEL_PATH = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/"
    "Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface"
)

# ──────────────────────────────────────────────────────────────
# Analysis parameters
# ──────────────────────────────────────────────────────────────
TOPK_SINGULAR_VECTORS = 64       # Cosine similarity heatmap에 사용할 상위 singular vector 수
ENERGY_THRESHOLDS = [0.90, 0.95, 0.99]  # Cumulative energy threshold for rank analysis
INCLUDE_EMBEDDING = False        # Embedding은 기본 제외 (tied weights + SVD peak 6.26GB)
SVD_DEVICE = "cuda:0"            # GPU SVD, OOM 시 CPU로 자동 fallback

# ──────────────────────────────────────────────────────────────
# Output paths
# ──────────────────────────────────────────────────────────────
ARTIFACT_DIR = Path("artifacts/svd_task_interference")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# ──────────────────────────────────────────────────────────────
# Plotting style
# ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", context="talk")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"Artifact dir: {ARTIFACT_DIR.resolve()}")

In [ ]:
"""Cell 2: State Dict Loading Utilities.

safetensors를 직접 사용하여 state dict만 로드한다.
AutoModelForCausalLM을 사용하지 않으므로 computation graph, optimizer state 등의
오버헤드가 없어 모델당 ~4GB (bf16)만 사용한다.
"""


def load_state_dicts_from_safetensors(model_dir: Path) -> Dict[str, torch.Tensor]:
    """safetensors 파일에서 state dict를 직접 로드한다.

    HuggingFace checkpoint 디렉토리의 index.json을 파싱하여 shard 파일을 찾고,
    각 shard를 순차적으로 로드하여 하나의 state dict으로 합친다.

    Args:
        model_dir: HuggingFace checkpoint 디렉토리 경로.
                   model.safetensors.index.json 또는 model.safetensors가 있어야 한다.

    Returns:
        Dict[str, torch.Tensor]: 파라미터 이름 -> 텐서 매핑.

    Raises:
        FileNotFoundError: safetensors 파일이 없는 경우.
    """
    # Multi-shard 케이스: index.json에서 shard 목록 추출
    index_path = model_dir / "model.safetensors.index.json"
    if index_path.exists():
        with open(index_path) as f:
            index = json.load(f)
        # weight_map: {param_name: shard_filename}
        shard_files = sorted(set(index["weight_map"].values()))
        state_dict = {}
        for shard_file in shard_files:
            shard_path = model_dir / shard_file
            state_dict.update(safetensors_load_file(str(shard_path)))
        return state_dict

    # Single-file 케이스
    single_path = model_dir / "model.safetensors"
    if single_path.exists():
        return safetensors_load_file(str(single_path))

    raise FileNotFoundError(f"No safetensors files found in {model_dir}")


def load_hf_state_dict(model_id: str) -> Dict[str, torch.Tensor]:
    """HuggingFace hub 모델의 state dict을 로드한다.

    transformers의 cached_file을 사용하여 로컬 캐시 경로를 resolve한 뒤
    safetensors로 직접 로드한다. 모델 그래프를 생성하지 않아 메모리 효율적이다.

    Args:
        model_id: HuggingFace hub model ID (예: 'Qwen/Qwen3-1.7B').

    Returns:
        Dict[str, torch.Tensor]: 파라미터 이름 -> 텐서 매핑.
    """
    from transformers.utils import cached_file

    # Multi-shard인 경우 index 파일로 캐시 디렉토리를 찾는다
    index_file = cached_file(
        model_id,
        "model.safetensors.index.json",
        _raise_exceptions_for_missing_entries=False,
    )
    if index_file is not None:
        model_dir = Path(index_file).parent
        return load_state_dicts_from_safetensors(model_dir)

    # Single-file 케이스
    single_file = cached_file(model_id, "model.safetensors")
    return safetensors_load_file(single_file)


print("Loading utilities defined.")

In [ ]:
"""Cell 3: Parameter Classification and SVD Utility Functions.

SVD 분석 대상 파라미터 필터링, 파라미터 이름 파싱,
GPU/CPU fallback SVD, effective rank, cumulative energy 계산을 정의한다.
"""

# ──────────────────────────────────────────────────────────────
# SVD 분석 대상 2D weight matrix suffixes
# 1D 파라미터 (LayerNorm, QK-norm 등)는 SVD에 의미가 없으므로 제외
# ──────────────────────────────────────────────────────────────
SVD_TARGET_SUFFIXES = [
    "self_attn.q_proj.weight",   # (2048, 2048)
    "self_attn.k_proj.weight",   # (1024, 2048)
    "self_attn.v_proj.weight",   # (1024, 2048)
    "self_attn.o_proj.weight",   # (2048, 2048)
    "mlp.gate_proj.weight",      # (6144, 2048)
    "mlp.up_proj.weight",        # (6144, 2048)
    "mlp.down_proj.weight",      # (2048, 6144)
]

# Projection 이름 -> block type 매핑
PROJ_TO_BLOCK = {
    "q_proj": "attn",
    "k_proj": "attn",
    "v_proj": "attn",
    "o_proj": "attn",
    "gate_proj": "mlp",
    "up_proj": "mlp",
    "down_proj": "mlp",
}

# 레이어 이름 파싱용 정규식
# 예: 'model.layers.14.self_attn.q_proj.weight' -> layer=14, proj=q_proj
_LAYER_PATTERN = re.compile(
    r"model\.layers\.(\d+)\.(self_attn|mlp)\.([a-z_]+)\.weight"
)


def is_svd_target(param_name: str, include_embedding: bool = False) -> bool:
    """SVD 분석 대상 파라미터인지 판별한다.

    2D weight matrix만 분석 대상으로 한다.
    1D 파라미터 (LayerNorm weight, QK-norm weight, bias 등)는 제외한다.

    Args:
        param_name: 파라미터의 fully qualified name.
        include_embedding: True이면 embed_tokens도 포함 (기본 False).

    Returns:
        bool: SVD 분석 대상이면 True.
    """
    if include_embedding and param_name in (
        "model.embed_tokens.weight",
        "lm_head.weight",
    ):
        return True
    return any(param_name.endswith(suffix) for suffix in SVD_TARGET_SUFFIXES)


def parse_param_identity(param_name: str) -> Tuple[int, str, str]:
    """파라미터 이름을 (layer_idx, block_type, proj_name)으로 파싱한다.

    예: 'model.layers.14.self_attn.q_proj.weight'
        -> (14, 'attn', 'q_proj')

    Args:
        param_name: 파라미터의 fully qualified name.

    Returns:
        Tuple[int, str, str]: (layer_idx, block_type, proj_name).
            - layer_idx: 0-27 for decoder layers, -1 for embedding/lm_head
            - block_type: 'attn', 'mlp', 'embed', or 'lm_head'
            - proj_name: e.g., 'q_proj', 'gate_proj'
    """
    # Embedding/LM head 특수 케이스
    if "embed_tokens" in param_name:
        return (-1, "embed", "embed_tokens")
    if "lm_head" in param_name:
        return (-1, "lm_head", "lm_head")

    # Decoder layer 파싱
    m = _LAYER_PATTERN.match(param_name)
    if m is None:
        raise ValueError(f"Cannot parse parameter name: {param_name}")

    layer_idx = int(m.group(1))
    module_type = m.group(2)  # 'self_attn' or 'mlp'
    proj_name = m.group(3)    # 'q_proj', 'gate_proj', etc.
    block_type = PROJ_TO_BLOCK.get(proj_name, module_type)

    return (layer_idx, block_type, proj_name)


def safe_svd_on_device(
    matrix: torch.Tensor, device: str = "cuda:0"
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """GPU에서 full SVD를 수행하고, OOM 발생 시 CPU로 fallback한다.

    torch.linalg.svd(full_matrices=False)를 사용하여 compact SVD를 수행한다.
    - U:  (m, k) left singular vectors
    - S:  (k,)   singular values (내림차순)
    - Vh: (k, n) right singular vectors (전치됨)
    여기서 k = min(m, n).

    Args:
        matrix: (m, n) 크기의 float32 텐서.
        device: SVD를 수행할 디바이스. 'cuda:0' 권장.

    Returns:
        Tuple[U, S, Vh]: 모두 CPU 텐서로 반환.
    """
    try:
        # GPU에서 SVD 수행 (cuSOLVER 사용, 수치적으로 안정적)
        matrix_dev = matrix.to(device)
        U, S, Vh = torch.linalg.svd(matrix_dev, full_matrices=False)
        result = (U.cpu(), S.cpu(), Vh.cpu())
        del matrix_dev  # GPU 메모리 즉시 해제
        return result
    except torch.cuda.OutOfMemoryError:
        # OOM 발생 시 GPU 캐시 정리 후 CPU에서 재시도 (LAPACK 사용)
        print(f"  ⚠ GPU OOM for shape {matrix.shape}, falling back to CPU")
        torch.cuda.empty_cache()
        U, S, Vh = torch.linalg.svd(matrix.cpu(), full_matrices=False)
        return U, S, Vh


def compute_effective_rank(singular_values: torch.Tensor, eps: float = 1e-12) -> float:
    """Shannon entropy 기반 effective rank를 계산한다.

    effective rank = exp(H(p))  여기서 p_i = s_i / sum(s), H = -sum(p log p)

    범위: [1, min(m, n)]
    - 값이 작을수록 정보가 소수의 singular vector에 집중 (low-rank)
    - 값이 클수록 정보가 분산 (full-rank에 가까움)

    Args:
        singular_values: (k,) 크기의 비음수 텐서.
        eps: 수치 안정성을 위한 epsilon.

    Returns:
        float: effective rank 값.
    """
    s = torch.clamp(singular_values, min=0.0)
    total = s.sum()
    if total <= eps:
        return 0.0
    # 정규화된 확률 분포
    p = s / total
    # Shannon entropy
    entropy = -torch.sum(p * torch.log(p + eps))
    return float(torch.exp(entropy).item())


def compute_cumulative_energy(singular_values: torch.Tensor) -> torch.Tensor:
    """Cumulative energy ratio를 계산한다.

    E(k) = sum(s[:k]^2) / sum(s^2)

    Frobenius norm 기준으로 상위 k개 singular value가 전체 에너지의
    몇 퍼센트를 차지하는지 계산한다.

    Args:
        singular_values: (k,) 크기의 비음수 텐서 (내림차순).

    Returns:
        torch.Tensor: (k,) 크기, 값 범위 [0, 1].
    """
    s_sq = singular_values ** 2
    total_energy = s_sq.sum()
    if total_energy <= 1e-12:
        return torch.zeros_like(singular_values)
    return torch.cumsum(s_sq, dim=0) / total_energy


def find_rank_for_energy(
    cumulative_energy: torch.Tensor, thresholds: List[float]
) -> Dict[float, int]:
    """각 energy threshold를 달성하는 최소 rank k를 찾는다.

    예: threshold=0.95이고 cumulative_energy[41] >= 0.95이면 rank=42 (1-indexed).

    Args:
        cumulative_energy: compute_cumulative_energy()의 출력.
        thresholds: 에너지 threshold 목록 (예: [0.90, 0.95, 0.99]).

    Returns:
        Dict[float, int]: threshold -> 최소 rank k (1-indexed) 매핑.
    """
    result = {}
    for thr in thresholds:
        indices = torch.where(cumulative_energy >= thr)[0]
        if len(indices) > 0:
            result[thr] = int(indices[0].item()) + 1  # 1-indexed
        else:
            result[thr] = int(len(cumulative_energy))
    return result


print("SVD utility functions defined.")
print(f"SVD target suffixes: {len(SVD_TARGET_SUFFIXES)} types per layer")
print(f"Expected total parameters: {len(SVD_TARGET_SUFFIXES)} × 28 layers = {len(SVD_TARGET_SUFFIXES) * 28}")

In [ ]:
"""Cell 4: Singular Task Interference (STI) Metric Computation.

논문의 STI 공식을 구현한다:
  STI({Δ_i}) = || (U^T U - I) Σ (V^T V - I) ||_1

U^T U의 off-diagonal block은 task 간 left singular vector의 내적(cosine similarity)을
나타내며, 직교성에서 벗어난 정도를 측정한다.
Σ를 가중치로 곱하여 영향력이 큰 방향에서의 충돌에 더 큰 페널티를 부여한다.
"""


def compute_sti_for_parameter(
    U_if: torch.Tensor,     # (m, k_if) left singular vectors of IF task
    S_if: torch.Tensor,     # (k_if,) singular values of IF task
    Vh_if: torch.Tensor,    # (k_if, n) right singular vectors of IF task (transposed)
    U_math: torch.Tensor,   # (m, k_math) left singular vectors of Math task
    S_math: torch.Tensor,   # (k_math,) singular values of Math task
    Vh_math: torch.Tensor,  # (k_math, n) right singular vectors of Math task (transposed)
    topk: Optional[int] = None,
) -> Dict[str, float]:
    """하나의 weight matrix에 대한 STI metric을 계산한다.

    STI 공식:
      U = [U_if[:, :topk] | U_math[:, :topk]]  (수평 결합)
      V = [V_if[:, :topk] | V_math[:, :topk]]  (수평 결합)
      Σ = block_diag(S_if[:topk], S_math[:topk])
      STI = || (U^T U - I) Σ (V^T V - I) ||_1

    topk=None이면 전체 singular vector를 사용한다 (full rank).

    L1 norm은 cross-task 오염의 전체 크기를 포착한다.
    Subspace가 직교하면 U^T U와 V^T V가 block diagonal identity가 되어 곱이 0이 된다.

    Args:
        U_if, S_if, Vh_if: IF task delta의 SVD 결과.
        U_math, S_math, Vh_math: Math task delta의 SVD 결과.
        topk: 상위 k개 singular vector만 사용. None이면 전체.

    Returns:
        Dict with:
          - 'sti': STI 값
          - 'mean_cross_cosine_left': left singular vector 간 평균 |cosine|
          - 'mean_cross_cosine_right': right singular vector 간 평균 |cosine|
          - 'max_cross_cosine_left': left singular vector 간 최대 |cosine|
          - 'max_cross_cosine_right': right singular vector 간 최대 |cosine|
          - 'k_if', 'k_math': 실제 사용된 rank 수
    """
    # Rank 결정: topk가 주어지면 truncate
    k_if = U_if.shape[1] if topk is None else min(topk, U_if.shape[1])
    k_math = U_math.shape[1] if topk is None else min(topk, U_math.shape[1])

    # Truncated singular vectors/values
    U_if_k = U_if[:, :k_if]          # (m, k_if)
    U_math_k = U_math[:, :k_math]    # (m, k_math)
    # Vh -> V (transpose): column-wise singular vectors
    V_if_k = Vh_if[:k_if, :].T       # (n, k_if)
    V_math_k = Vh_math[:k_math, :].T # (n, k_math)
    S_if_k = S_if[:k_if]
    S_math_k = S_math[:k_math]

    # 수평 결합: U = [U_if | U_math], shape (m, K) where K = k_if + k_math
    U_cat = torch.cat([U_if_k, U_math_k], dim=1)
    V_cat = torch.cat([V_if_k, V_math_k], dim=1)

    # Block diagonal Σ matrix
    S_cat = torch.cat([S_if_k, S_math_k])  # (K,)
    Sigma = torch.diag(S_cat)               # (K, K)

    # Gram matrices: U^T U, V^T V
    # 직교 subspace면 각각 identity matrix
    K = k_if + k_math
    I_K = torch.eye(K, dtype=U_cat.dtype)
    UU = U_cat.T @ U_cat  # (K, K)
    VV = V_cat.T @ V_cat  # (K, K)

    # 간섭 행렬: 직교성에서의 편차
    left_interference = UU - I_K
    right_interference = VV - I_K

    # STI = || left_interference @ Σ @ right_interference ||_1
    # L1 norm: 모든 원소의 절대값 합
    sti_matrix = left_interference @ Sigma @ right_interference
    sti = float(torch.sum(torch.abs(sti_matrix)).item())

    # ──────────────────────────────────────────────────────────
    # 진단용 부가 지표: cross-task cosine similarity
    # UU의 off-diagonal block [0:k_if, k_if:K]가 task 간 left vector overlap
    # ──────────────────────────────────────────────────────────
    UU_cross = UU[:k_if, k_if:]  # (k_if, k_math)
    VV_cross = VV[:k_if, k_if:]  # (k_if, k_math)

    mean_cross_cos_left = float(torch.abs(UU_cross).mean().item())
    mean_cross_cos_right = float(torch.abs(VV_cross).mean().item())
    max_cross_cos_left = float(torch.abs(UU_cross).max().item())
    max_cross_cos_right = float(torch.abs(VV_cross).max().item())

    return {
        "sti": sti,
        "mean_cross_cosine_left": mean_cross_cos_left,
        "mean_cross_cosine_right": mean_cross_cos_right,
        "max_cross_cosine_left": max_cross_cos_left,
        "max_cross_cosine_right": max_cross_cos_right,
        "k_if": k_if,
        "k_math": k_math,
    }


print("STI computation function defined.")

In [ ]:
"""Cell 5: Main SVD Analysis Loop.

196개 파라미터(7 weight types × 28 layers)를 순회하며
SVD 분해, rank 분석, STI 계산, cross-cosine matrix 저장을 수행한다.

메모리 관리: 각 파라미터 처리 후 즉시 중간 텐서를 해제한다.
Peak GPU 메모리: ~0.3GB (가장 큰 gate/up/down_proj 기준).
"""


@dataclass
class ParameterSVDResult:
    """하나의 weight matrix에 대한 SVD 분석 결과를 저장하는 데이터 클래스.

    Attributes:
        param_name: 파라미터의 fully qualified name.
        layer_idx: 레이어 인덱스 (0-27), embedding은 -1.
        block_type: 'attn' 또는 'mlp'.
        proj_name: projection 이름 (예: 'q_proj', 'gate_proj').
        shape: 원본 weight matrix의 shape.
        min_dim: min(m, n), SVD의 singular value 개수.

        singular_values_if: IF task delta의 singular values.
        effective_rank_if: IF task의 effective rank (Shannon entropy 기반).
        rank_90/95/99_if: 90/95/99% 에너지를 달성하는 최소 rank.

        singular_values_math: Math task delta의 singular values.
        effective_rank_math: Math task의 effective rank.
        rank_90/95/99_math: 90/95/99% 에너지를 달성하는 최소 rank.

        sti: Full-rank STI 값.
        sti_topk: Top-k truncated STI 값.
        mean/max_cross_cosine_left/right: Cross-task cosine similarity 통계.

        cross_cosine_matrix_left: (topk, topk) left singular vector 간 cosine.
        cross_cosine_matrix_right: (topk, topk) right singular vector 간 cosine.
    """
    param_name: str
    layer_idx: int
    block_type: str
    proj_name: str
    shape: Tuple[int, int]
    min_dim: int

    # IF task SVD
    singular_values_if: np.ndarray
    effective_rank_if: float
    rank_90_if: int
    rank_95_if: int
    rank_99_if: int

    # Math task SVD
    singular_values_math: np.ndarray
    effective_rank_math: float
    rank_90_math: int
    rank_95_math: int
    rank_99_math: int

    # Cross-task interference
    sti: float
    sti_topk: float
    mean_cross_cosine_left: float
    mean_cross_cosine_right: float
    max_cross_cosine_left: float
    max_cross_cosine_right: float

    # Heatmap용 cross-cosine matrices (top-k × top-k)
    cross_cosine_matrix_left: Optional[np.ndarray] = None
    cross_cosine_matrix_right: Optional[np.ndarray] = None


def run_svd_analysis(
    base_sd: Dict[str, torch.Tensor],
    if_sd: Dict[str, torch.Tensor],
    math_sd: Dict[str, torch.Tensor],
    topk: int = 64,
    energy_thresholds: List[float] = None,
    include_embedding: bool = False,
    svd_device: str = "cuda:0",
) -> List[ParameterSVDResult]:
    """모든 대상 weight matrix에 대해 SVD 분석을 수행한다.

    각 2D weight 파라미터에 대해:
    1. Task vector 계산: delta = ft_sd[key].float() - base_sd[key].float()
    2. SVD 분해 (GPU, OOM시 CPU fallback)
    3. Singular value spectrum 분석 (effective rank, energy thresholds)
    4. STI metric 계산 (full + top-k)
    5. Cross-cosine matrix 저장 (top-k × top-k, heatmap용)

    메모리 관리: 각 파라미터 처리 후 중간 텐서를 즉시 해제한다.

    Args:
        base_sd: Base model state dict.
        if_sd: IF fine-tuned model state dict.
        math_sd: Math fine-tuned model state dict.
        topk: Cross-cosine heatmap 및 truncated STI에 사용할 상위 singular vector 수.
        energy_thresholds: Cumulative energy threshold 목록.
        include_embedding: Embedding layer 포함 여부.
        svd_device: SVD 수행 디바이스.

    Returns:
        List[ParameterSVDResult]: 파라미터별 분석 결과 목록.
    """
    if energy_thresholds is None:
        energy_thresholds = [0.90, 0.95, 0.99]

    results = []

    # SVD 분석 대상 파라미터 필터링
    target_keys = sorted(
        [k for k in base_sd.keys() if is_svd_target(k, include_embedding)]
    )
    print(f"Total SVD target parameters: {len(target_keys)}")

    for param_name in tqdm(target_keys, desc="SVD analysis"):
        # ── 1. Key 존재 및 shape 일치 검증 ──
        assert param_name in if_sd, f"Missing in IF state dict: {param_name}"
        assert param_name in math_sd, f"Missing in Math state dict: {param_name}"
        assert base_sd[param_name].shape == if_sd[param_name].shape, (
            f"Shape mismatch for {param_name}: "
            f"base={base_sd[param_name].shape} vs if={if_sd[param_name].shape}"
        )

        layer_idx, block_type, proj_name = parse_param_identity(param_name)
        shape = tuple(base_sd[param_name].shape)
        min_dim = min(shape)

        # ── 2. Task vector 계산 (float32로 upcast) ──
        base_fp32 = base_sd[param_name].to(torch.float32)
        delta_if = if_sd[param_name].to(torch.float32) - base_fp32
        delta_math = math_sd[param_name].to(torch.float32) - base_fp32
        del base_fp32  # 즉시 해제

        # ── 3. SVD 분해 ──
        U_if, S_if, Vh_if = safe_svd_on_device(delta_if, device=svd_device)
        U_math, S_math, Vh_math = safe_svd_on_device(delta_math, device=svd_device)
        del delta_if, delta_math  # 원본 delta 해제

        # ── 4. Rank Analysis: IF task ──
        cum_energy_if = compute_cumulative_energy(S_if)
        ranks_if = find_rank_for_energy(cum_energy_if, energy_thresholds)
        eff_rank_if = compute_effective_rank(S_if)

        # ── 5. Rank Analysis: Math task ──
        cum_energy_math = compute_cumulative_energy(S_math)
        ranks_math = find_rank_for_energy(cum_energy_math, energy_thresholds)
        eff_rank_math = compute_effective_rank(S_math)

        # ── 6. STI computation (full rank) ──
        sti_full = compute_sti_for_parameter(
            U_if, S_if, Vh_if, U_math, S_math, Vh_math, topk=None
        )

        # ── 7. STI computation (top-k truncated) ──
        sti_topk_result = compute_sti_for_parameter(
            U_if, S_if, Vh_if, U_math, S_math, Vh_math, topk=topk
        )

        # ── 8. Cross-cosine matrices for heatmap (top-k × top-k) ──
        k = min(topk, min_dim)
        # Left: U_if^T @ U_math (top-k singular vectors 간 cosine similarity)
        cross_left = (U_if[:, :k].T @ U_math[:, :k]).numpy()
        # Right: V_if^T @ V_math = (Vh_if[:k] @ Vh_math[:k]^T)^T 이지만
        # 직접 Vh 행으로 계산: Vh_if[:k] @ Vh_math[:k].T
        cross_right = (Vh_if[:k, :] @ Vh_math[:k, :].T).numpy()

        # ── 9. 결과 저장 ──
        result = ParameterSVDResult(
            param_name=param_name,
            layer_idx=layer_idx,
            block_type=block_type,
            proj_name=proj_name,
            shape=shape,
            min_dim=min_dim,
            # IF task
            singular_values_if=S_if.numpy(),
            effective_rank_if=eff_rank_if,
            rank_90_if=ranks_if[0.90],
            rank_95_if=ranks_if[0.95],
            rank_99_if=ranks_if[0.99],
            # Math task
            singular_values_math=S_math.numpy(),
            effective_rank_math=eff_rank_math,
            rank_90_math=ranks_math[0.90],
            rank_95_math=ranks_math[0.95],
            rank_99_math=ranks_math[0.99],
            # Interference
            sti=sti_full["sti"],
            sti_topk=sti_topk_result["sti"],
            mean_cross_cosine_left=sti_topk_result["mean_cross_cosine_left"],
            mean_cross_cosine_right=sti_topk_result["mean_cross_cosine_right"],
            max_cross_cosine_left=sti_topk_result["max_cross_cosine_left"],
            max_cross_cosine_right=sti_topk_result["max_cross_cosine_right"],
            # Heatmap matrices
            cross_cosine_matrix_left=cross_left,
            cross_cosine_matrix_right=cross_right,
        )
        results.append(result)

        # ── 10. 메모리 정리 ──
        del U_if, S_if, Vh_if, U_math, S_math, Vh_math
        del cum_energy_if, cum_energy_math

    return results


print("Main analysis loop defined.")

In [ ]:
"""Cell 6: Execute SVD Analysis.

3개 모델의 state dict을 로드하고 SVD 분석을 실행한 뒤 state dict을 해제한다.
예상 소요 시간: ~10-15분 (GPU SVD 기준).
"""

# ── State dict 로드 ──
print("=" * 60)
print("Loading state dictionaries (safetensors only, no model graph)")
print("=" * 60)

print("\n[1/3] Loading base model state dict...")
base_sd = load_hf_state_dict(BASE_MODEL_ID)
print(f"  Loaded {len(base_sd)} keys")

print("\n[2/3] Loading IF model state dict...")
if_sd = load_state_dicts_from_safetensors(IF_MODEL_PATH)
print(f"  Loaded {len(if_sd)} keys")

print("\n[3/3] Loading Math model state dict...")
math_sd = load_state_dicts_from_safetensors(MATH_MODEL_PATH)
print(f"  Loaded {len(math_sd)} keys")

# ── SVD 분석 실행 ──
print("\n" + "=" * 60)
print("Running SVD analysis on all target weight matrices")
print("=" * 60)

svd_results = run_svd_analysis(
    base_sd=base_sd,
    if_sd=if_sd,
    math_sd=math_sd,
    topk=TOPK_SINGULAR_VECTORS,
    energy_thresholds=ENERGY_THRESHOLDS,
    include_embedding=INCLUDE_EMBEDDING,
    svd_device=SVD_DEVICE,
)

# ── State dict 메모리 해제 ──
del base_sd, if_sd, math_sd
gc.collect()
torch.cuda.empty_cache()

print(f"\nCompleted SVD analysis for {len(svd_results)} weight matrices.")

In [ ]:
"""Cell 7: Build Results DataFrames and Save CSVs.

파라미터별 결과를 DataFrame으로 변환하고,
layer별 집계 DataFrame을 생성하여 CSV로 저장한다.
"""


def build_parameter_dataframe(results: List[ParameterSVDResult]) -> pd.DataFrame:
    """파라미터별 SVD 분석 결과를 tidy DataFrame으로 변환한다.

    Args:
        results: run_svd_analysis()의 출력.

    Returns:
        pd.DataFrame: 196행 (7 proj types × 28 layers), 주요 지표 포함.
    """
    rows = []
    for r in results:
        rows.append({
            "param_name": r.param_name,
            "layer_idx": r.layer_idx,
            "block_type": r.block_type,
            "proj_name": r.proj_name,
            "shape": str(r.shape),
            "min_dim": r.min_dim,
            # IF task rank analysis
            "effective_rank_if": r.effective_rank_if,
            "rank_90_if": r.rank_90_if,
            "rank_95_if": r.rank_95_if,
            "rank_99_if": r.rank_99_if,
            # Math task rank analysis
            "effective_rank_math": r.effective_rank_math,
            "rank_90_math": r.rank_90_math,
            "rank_95_math": r.rank_95_math,
            "rank_99_math": r.rank_99_math,
            # Interference metrics
            "sti": r.sti,
            "sti_topk": r.sti_topk,
            "mean_cross_cos_left": r.mean_cross_cosine_left,
            "mean_cross_cos_right": r.mean_cross_cosine_right,
            "max_cross_cos_left": r.max_cross_cosine_left,
            "max_cross_cos_right": r.max_cross_cosine_right,
            # Rank ratio: 99% rank를 min_dim으로 나눈 값 (low-rank 정도)
            "rank_99_ratio_if": r.rank_99_if / r.min_dim,
            "rank_99_ratio_math": r.rank_99_math / r.min_dim,
        })
    return pd.DataFrame(rows)


def build_layer_dataframe(param_df: pd.DataFrame) -> pd.DataFrame:
    """파라미터별 결과를 layer 단위로 집계한다.

    각 layer에 대해 attention과 MLP block을 분리하여 집계하고,
    전체 layer 합산도 포함한다.

    Args:
        param_df: build_parameter_dataframe()의 출력.

    Returns:
        pd.DataFrame: layer별 집계 결과.
    """
    # Layer 전체 집계 (attn + mlp 합산)
    layer_agg = param_df.groupby("layer_idx").agg({
        # STI: layer 내 모든 파라미터의 합 (전체 간섭량)
        "sti": "sum",
        "sti_topk": "sum",
        # Effective rank: 평균
        "effective_rank_if": "mean",
        "effective_rank_math": "mean",
        # 99% rank ratio: 평균 (low-rank 정도의 대표값)
        "rank_99_ratio_if": "mean",
        "rank_99_ratio_math": "mean",
        # Cross-cosine: 평균 및 최대
        "mean_cross_cos_left": "mean",
        "mean_cross_cos_right": "mean",
        "max_cross_cos_left": "max",
        "max_cross_cos_right": "max",
    }).reset_index()

    # Block type별 집계 (attn vs mlp 분리)
    block_agg = param_df.groupby(["layer_idx", "block_type"]).agg({
        "sti": "sum",
        "effective_rank_if": "mean",
        "effective_rank_math": "mean",
        "mean_cross_cos_left": "mean",
        "max_cross_cos_left": "max",
    }).reset_index()

    # Attn/MLP STI를 별도 컬럼으로 추가
    attn_sti = block_agg[block_agg["block_type"] == "attn"][["layer_idx", "sti"]].rename(
        columns={"sti": "sti_attn"}
    )
    mlp_sti = block_agg[block_agg["block_type"] == "mlp"][["layer_idx", "sti"]].rename(
        columns={"sti": "sti_mlp"}
    )
    layer_agg = layer_agg.merge(attn_sti, on="layer_idx", how="left")
    layer_agg = layer_agg.merge(mlp_sti, on="layer_idx", how="left")

    return layer_agg


# ── DataFrame 구성 ──
param_df = build_parameter_dataframe(svd_results)
layer_df = build_layer_dataframe(param_df)

# ── CSV 저장 ──
param_csv_path = ARTIFACT_DIR / "parameter_svd_metrics.csv"
layer_csv_path = ARTIFACT_DIR / "layer_svd_metrics.csv"
param_df.to_csv(param_csv_path, index=False)
layer_df.to_csv(layer_csv_path, index=False)

print(f"Saved parameter metrics: {param_csv_path}")
print(f"Saved layer metrics: {layer_csv_path}")

# ── 요약 출력 ──
print("\n" + "=" * 60)
print("Summary Statistics")
print("=" * 60)
print(f"\nTotal parameters analyzed: {len(param_df)}")
print(f"\n--- STI ---")
print(f"  Mean STI (full):  {param_df['sti'].mean():.4f}")
print(f"  Max STI (full):   {param_df['sti'].max():.4f}  ({param_df.loc[param_df['sti'].idxmax(), 'param_name']})")
print(f"  Mean STI (topk):  {param_df['sti_topk'].mean():.4f}")
print(f"\n--- Effective Rank ---")
print(f"  IF mean:   {param_df['effective_rank_if'].mean():.1f}")
print(f"  Math mean: {param_df['effective_rank_math'].mean():.1f}")
print(f"\n--- 99% Energy Rank Ratio (rank / min_dim) ---")
print(f"  IF mean:   {param_df['rank_99_ratio_if'].mean():.4f} ({param_df['rank_99_ratio_if'].mean()*100:.1f}%)")
print(f"  Math mean: {param_df['rank_99_ratio_math'].mean():.4f} ({param_df['rank_99_ratio_math'].mean()*100:.1f}%)")

display(param_df.head(14))  # 처음 2개 layer (7 proj × 2)

In [ ]:
"""Cell 8: Analysis A — Singular Value Spectrum.

Singular value의 감소 속도를 시각화하여 low-rank 여부를 판단한다.
빠르게 감소할수록 정보가 소수의 singular vector에 집중되어 low-rank이다.

Plot 1: 대표 layer별 IF vs Math singular value spectrum (log scale)
Plot 2: Effective rank heatmap (28 layers × 7 projection types)
"""

# ── Plot 1: Singular value spectrum for representative layers ──
# Layer 0 (earliest), 7, 14 (middle), 21, 27 (deepest)을 선택
selected_layers = [0, 7, 14, 21, 27]
proj_types = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

fig, axes = plt.subplots(
    len(proj_types), len(selected_layers),
    figsize=(28, 32),
    constrained_layout=True,
)
fig.suptitle(
    "Singular Value Spectrum: IF (blue) vs Math (red)\n"
    "X-axis: singular value index (rank), Y-axis: singular value (log scale)",
    fontsize=18, fontweight="bold", y=1.01,
)

# 결과를 (layer_idx, proj_name)으로 인덱싱하기 위한 딕셔너리
result_map = {(r.layer_idx, r.proj_name): r for r in svd_results}

for row_idx, proj in enumerate(proj_types):
    for col_idx, layer in enumerate(selected_layers):
        ax = axes[row_idx, col_idx]
        key = (layer, proj)

        if key not in result_map:
            ax.set_visible(False)
            continue

        r = result_map[key]
        x = np.arange(1, len(r.singular_values_if) + 1)

        # IF task: 파란색, Math task: 빨간색
        ax.semilogy(x, r.singular_values_if, color="steelblue", alpha=0.8, linewidth=1.5, label="IF")
        ax.semilogy(x, r.singular_values_math, color="indianred", alpha=0.8, linewidth=1.5, label="Math")

        # 99% 에너지 rank를 수직선으로 표시
        ax.axvline(r.rank_99_if, color="steelblue", linestyle="--", alpha=0.5, linewidth=1)
        ax.axvline(r.rank_99_math, color="indianred", linestyle="--", alpha=0.5, linewidth=1)

        ax.set_title(f"L{layer} {proj}\nrank99: IF={r.rank_99_if}, Math={r.rank_99_math}", fontsize=10)

        if col_idx == 0:
            ax.set_ylabel(proj, fontsize=12, fontweight="bold")
        if row_idx == 0 and col_idx == 0:
            ax.legend(fontsize=8)

fig.savefig(ARTIFACT_DIR / "singular_value_spectra.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR / 'singular_value_spectra.png'}")


# ── Plot 2: Effective rank heatmap ──
fig, axes = plt.subplots(1, 2, figsize=(24, 10), constrained_layout=True)
fig.suptitle("Effective Rank (Shannon Entropy) by Layer × Projection Type", fontsize=16, fontweight="bold")

for ax_idx, (task_name, col_name) in enumerate(
    [("IF Task", "effective_rank_if"), ("Math Task", "effective_rank_math")]
):
    ax = axes[ax_idx]
    # Pivot: rows=layer_idx, columns=proj_name
    pivot = param_df.pivot(index="layer_idx", columns="proj_name", values=col_name)
    # Reorder columns
    pivot = pivot.reindex(columns=proj_types)
    sns.heatmap(
        pivot, ax=ax, cmap="YlOrRd", annot=True, fmt=".0f",
        linewidths=0.5, cbar_kws={"label": "Effective Rank"},
    )
    ax.set_title(task_name, fontsize=14)
    ax.set_xlabel("Projection Type")
    ax.set_ylabel("Layer Index")

fig.savefig(ARTIFACT_DIR / "effective_rank_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR / 'effective_rank_heatmap.png'}")

In [ ]:
"""Cell 9: Analysis B — Cross-Task Cosine Similarity Heatmaps.

IF와 Math task의 singular vector 간 cosine similarity를 시각화한다.
U_if^T @ U_math (left)와 V_if^T @ V_math (right)의 top-k × top-k heatmap.

높은 off-diagonal 값은 서로 다른 singular component가 task 간에 간섭함을 의미한다.
STI가 가장 높은 3개와 가장 낮은 3개 파라미터를 비교한다.
"""

# ── Plot 1: High-STI vs Low-STI 파라미터의 cross-cosine heatmap ──
sorted_by_sti = sorted(svd_results, key=lambda r: r.sti, reverse=True)
high_sti_params = sorted_by_sti[:3]   # STI 상위 3개
low_sti_params = sorted_by_sti[-3:]   # STI 하위 3개
selected_params = high_sti_params + low_sti_params

fig, axes = plt.subplots(
    len(selected_params), 2,
    figsize=(18, len(selected_params) * 4.5),
    constrained_layout=True,
)
fig.suptitle(
    "Cross-Task Singular Vector Cosine Similarity\n"
    f"Top-{TOPK_SINGULAR_VECTORS} vectors | Top 3 High-STI (rows 1-3) vs Top 3 Low-STI (rows 4-6)",
    fontsize=16, fontweight="bold",
)

for row_idx, r in enumerate(selected_params):
    # Left singular vectors: U_if^T @ U_math
    ax_left = axes[row_idx, 0]
    sns.heatmap(
        r.cross_cosine_matrix_left,
        ax=ax_left, cmap="RdBu_r", vmin=-1, vmax=1,
        xticklabels=False, yticklabels=False,
        cbar_kws={"label": "cosine sim"},
    )
    label = "HIGH" if row_idx < 3 else "LOW"
    ax_left.set_title(
        f"[{label}] L{r.layer_idx} {r.proj_name} | Left (U_if^T @ U_math)\n"
        f"STI={r.sti:.2f}, mean|cos|={r.mean_cross_cosine_left:.4f}",
        fontsize=10,
    )
    ax_left.set_xlabel("Math SV index")
    ax_left.set_ylabel("IF SV index")

    # Right singular vectors: V_if^T @ V_math (= Vh_if @ Vh_math^T)
    ax_right = axes[row_idx, 1]
    sns.heatmap(
        r.cross_cosine_matrix_right,
        ax=ax_right, cmap="RdBu_r", vmin=-1, vmax=1,
        xticklabels=False, yticklabels=False,
        cbar_kws={"label": "cosine sim"},
    )
    ax_right.set_title(
        f"[{label}] L{r.layer_idx} {r.proj_name} | Right (V_if^T @ V_math)\n"
        f"STI={r.sti:.2f}, mean|cos|={r.mean_cross_cosine_right:.4f}",
        fontsize=10,
    )
    ax_right.set_xlabel("Math SV index")
    ax_right.set_ylabel("IF SV index")

fig.savefig(ARTIFACT_DIR / "cross_cosine_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR / 'cross_cosine_heatmaps.png'}")


# ── Plot 2: Layer별 mean |cross-cosine| 추이 ──
fig, axes = plt.subplots(1, 2, figsize=(22, 7), constrained_layout=True)
fig.suptitle("Mean |Cross-Cosine Similarity| by Layer", fontsize=16, fontweight="bold")

for ax_idx, (title, left_col, right_col) in enumerate([
    ("Mean |cos| (averaged across projections)", "mean_cross_cos_left", "mean_cross_cos_right"),
    ("Max |cos| (worst-case across projections)", "max_cross_cos_left", "max_cross_cos_right"),
]):
    ax = axes[ax_idx]

    # Layer별 집계
    if "mean" in title.lower():
        layer_left = param_df.groupby("layer_idx")[left_col].mean()
        layer_right = param_df.groupby("layer_idx")[right_col].mean()
    else:
        layer_left = param_df.groupby("layer_idx")[left_col].max()
        layer_right = param_df.groupby("layer_idx")[right_col].max()

    ax.plot(layer_left.index, layer_left.values, "o-", color="steelblue",
            linewidth=2, markersize=5, label="Left (U)")
    ax.plot(layer_right.index, layer_right.values, "s-", color="indianred",
            linewidth=2, markersize=5, label="Right (V)")

    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Layer Index")
    ax.set_ylabel("|Cosine Similarity|")
    ax.legend()
    ax.set_xlim(-0.5, 27.5)

fig.savefig(ARTIFACT_DIR / "cross_cosine_by_layer.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR / 'cross_cosine_by_layer.png'}")

In [ ]:
"""Cell 10: Analysis C — STI (Singular Task Interference) Per Layer.

Layer depth에 따른 STI 변화를 시각화한다.

기대 패턴 (논문 기반):
- Early layers (layer 0-5): 높은 STI → 일반적 feature 공유로 인한 간섭
- Deep layers (layer 22-27): 낮은 STI → task-specific feature로 직교에 가까움
- Transformer block 내부에서 attention → MLP 패턴 변화 가능
"""

fig, axes = plt.subplots(2, 2, figsize=(24, 16), constrained_layout=True)
fig.suptitle(
    "Singular Task Interference (STI) Analysis by Layer Depth",
    fontsize=18, fontweight="bold",
)

# ── Plot 1 (top-left): STI vs Layer Depth (Attention vs MLP) ──
ax = axes[0, 0]
ax.plot(
    layer_df["layer_idx"], layer_df["sti_attn"],
    "o-", color="steelblue", linewidth=2, markersize=6, label="Attention STI",
)
ax.plot(
    layer_df["layer_idx"], layer_df["sti_mlp"],
    "s-", color="indianred", linewidth=2, markersize=6, label="MLP STI",
)
ax.plot(
    layer_df["layer_idx"], layer_df["sti"],
    "D-", color="dimgray", linewidth=2.5, markersize=7, label="Total STI",
)
ax.set_title("STI vs Layer Depth", fontsize=14)
ax.set_xlabel("Layer Index")
ax.set_ylabel("STI (sum over projections)")
ax.legend()
ax.set_xlim(-0.5, 27.5)

# ── Plot 2 (top-right): Per-Projection STI Heatmap ──
ax = axes[0, 1]
pivot_sti = param_df.pivot(index="layer_idx", columns="proj_name", values="sti")
pivot_sti = pivot_sti.reindex(columns=proj_types)
sns.heatmap(
    pivot_sti, ax=ax, cmap="YlOrRd", annot=True, fmt=".1f",
    linewidths=0.5, cbar_kws={"label": "STI"},
)
ax.set_title("Per-Projection STI Heatmap", fontsize=14)
ax.set_xlabel("Projection Type")
ax.set_ylabel("Layer Index")

# ── Plot 3 (bottom-left): STI(full) vs STI(topk) 비교 ──
ax = axes[1, 0]
ax.scatter(
    param_df["sti"], param_df["sti_topk"],
    c=param_df["layer_idx"], cmap="viridis", alpha=0.7, s=40, edgecolors="gray",
)
# y=x 참조선
max_val = max(param_df["sti"].max(), param_df["sti_topk"].max())
ax.plot([0, max_val], [0, max_val], "k--", alpha=0.5, label="y=x")
ax.set_title(f"STI (full rank) vs STI (top-{TOPK_SINGULAR_VECTORS})", fontsize=14)
ax.set_xlabel("STI (full rank)")
ax.set_ylabel(f"STI (top-{TOPK_SINGULAR_VECTORS})")
cbar = plt.colorbar(ax.collections[0], ax=ax)
cbar.set_label("Layer Index")
ax.legend()

# ── Plot 4 (bottom-right): Top-10 highest STI parameters ──
ax = axes[1, 1]
top10 = param_df.nlargest(10, "sti")
labels = [f"L{r.layer_idx} {r.proj_name}" for _, r in top10.iterrows()]
colors = ["steelblue" if bt == "attn" else "indianred" for bt in top10["block_type"]]
ax.barh(range(len(top10)), top10["sti"].values, color=colors, edgecolor="gray")
ax.set_yticks(range(len(top10)))
ax.set_yticklabels(labels, fontsize=11)
ax.invert_yaxis()  # 가장 높은 STI를 위에 표시
ax.set_title("Top-10 Highest STI Parameters", fontsize=14)
ax.set_xlabel("STI")
# 범례: attn=blue, mlp=red
from matplotlib.patches import Patch
ax.legend(
    handles=[Patch(color="steelblue", label="Attention"), Patch(color="indianred", label="MLP")],
    loc="lower right",
)

fig.savefig(ARTIFACT_DIR / "sti_by_layer.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR / 'sti_by_layer.png'}")

In [ ]:
"""Cell 11: Analysis D — Cumulative Energy and Low-Rank Reconstruction.

상위 k개 singular value로 전체 에너지의 몇 %를 재구성할 수 있는지 분석한다.

핵심 질문:
- Task matrix Δ_i가 본질적으로 low-rank인가?
- 상위 3~10%의 singular vector로 99% 재구성이 가능한가?
- IF와 Math task의 low-rank 특성이 다른가?
"""

# ── Plot 1: Cumulative energy curves for representative parameters ──
selected_layers_energy = [0, 14, 27]  # Early, middle, deep
selected_projs_energy = ["q_proj", "gate_proj"]  # Attn 대표, MLP 대표

fig, axes = plt.subplots(
    len(selected_projs_energy), len(selected_layers_energy),
    figsize=(22, 12),
    constrained_layout=True,
)
fig.suptitle(
    "Cumulative Energy: sum(s[:k]²) / sum(s²)\n"
    "IF (blue) vs Math (red) | Dashed lines at 90%, 95%, 99%",
    fontsize=16, fontweight="bold",
)

for row_idx, proj in enumerate(selected_projs_energy):
    for col_idx, layer in enumerate(selected_layers_energy):
        ax = axes[row_idx, col_idx]
        key = (layer, proj)

        if key not in result_map:
            ax.set_visible(False)
            continue

        r = result_map[key]
        # Cumulative energy 재계산 (singular values에서)
        s_if = torch.from_numpy(r.singular_values_if)
        s_math = torch.from_numpy(r.singular_values_math)
        cum_if = compute_cumulative_energy(s_if).numpy()
        cum_math = compute_cumulative_energy(s_math).numpy()
        x = np.arange(1, len(cum_if) + 1)

        ax.plot(x, cum_if, color="steelblue", linewidth=2, label="IF")
        ax.plot(x, cum_math, color="indianred", linewidth=2, label="Math")

        # Energy threshold 수평선
        for thr, ls in [(0.90, ":"), (0.95, "-."), (0.99, "--")]:
            ax.axhline(thr, color="gray", linestyle=ls, alpha=0.5, linewidth=1)
            ax.text(len(x) * 0.85, thr + 0.005, f"{thr:.0%}", fontsize=8, color="gray")

        # 99% rank 수직 마커
        ax.axvline(r.rank_99_if, color="steelblue", linestyle="--", alpha=0.4)
        ax.axvline(r.rank_99_math, color="indianred", linestyle="--", alpha=0.4)

        ax.set_title(
            f"L{layer} {proj} (dim={r.min_dim})\n"
            f"99% rank: IF={r.rank_99_if} ({r.rank_99_if/r.min_dim*100:.1f}%), "
            f"Math={r.rank_99_math} ({r.rank_99_math/r.min_dim*100:.1f}%)",
            fontsize=10,
        )
        ax.set_xlabel("Rank k")
        ax.set_ylabel("Cumulative Energy")
        ax.set_ylim(0, 1.05)
        if row_idx == 0 and col_idx == 0:
            ax.legend(fontsize=9)

fig.savefig(ARTIFACT_DIR / "cumulative_energy_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR / 'cumulative_energy_curves.png'}")


# ── Plot 2: Rank requirement heatmaps (90%, 95%, 99%) ──
fig, axes = plt.subplots(2, 3, figsize=(30, 18), constrained_layout=True)
fig.suptitle(
    "Minimum Rank for Energy Reconstruction\n"
    "Top row: IF Task | Bottom row: Math Task",
    fontsize=16, fontweight="bold",
)

for task_idx, (task_label, suffix) in enumerate([("IF", "_if"), ("Math", "_math")]):
    for thr_idx, thr in enumerate([90, 95, 99]):
        ax = axes[task_idx, thr_idx]
        col_name = f"rank_{thr}{suffix}"
        pivot = param_df.pivot(index="layer_idx", columns="proj_name", values=col_name)
        pivot = pivot.reindex(columns=proj_types)
        sns.heatmap(
            pivot, ax=ax, cmap="YlGnBu", annot=True, fmt="d",
            linewidths=0.5, cbar_kws={"label": "Rank"},
        )
        ax.set_title(f"{task_label} Task — {thr}% Energy Rank", fontsize=13)
        ax.set_xlabel("Projection Type")
        ax.set_ylabel("Layer Index")

fig.savefig(ARTIFACT_DIR / "rank_requirement_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR / 'rank_requirement_heatmaps.png'}")


# ── Plot 3: Effective rank scatter (IF vs Math) ──
fig, ax = plt.subplots(1, 1, figsize=(10, 10), constrained_layout=True)

scatter = ax.scatter(
    param_df["effective_rank_if"],
    param_df["effective_rank_math"],
    c=param_df["layer_idx"],
    cmap="viridis",
    s=60,
    alpha=0.7,
    edgecolors="gray",
)

# y=x 참조선
max_rank = max(param_df["effective_rank_if"].max(), param_df["effective_rank_math"].max())
ax.plot([0, max_rank], [0, max_rank], "k--", alpha=0.5, label="y=x")

# Projection type별 마커 어노테이션 (outlier만)
for _, row in param_df.iterrows():
    diff = abs(row["effective_rank_if"] - row["effective_rank_math"])
    if diff > 50:  # 큰 차이가 있는 파라미터만 라벨
        ax.annotate(
            f"L{row['layer_idx']} {row['proj_name']}",
            (row["effective_rank_if"], row["effective_rank_math"]),
            fontsize=7, alpha=0.8,
        )

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Layer Index")
ax.set_title("Effective Rank: IF vs Math\n(colored by layer depth)", fontsize=14)
ax.set_xlabel("Effective Rank (IF Task)")
ax.set_ylabel("Effective Rank (Math Task)")
ax.legend()
ax.set_aspect("equal")

fig.savefig(ARTIFACT_DIR / "effective_rank_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR / 'effective_rank_scatter.png'}")

In [ ]:
"""Cell 12: Summary Dashboard.

4-panel 종합 figure로 핵심 분석 결과를 한 눈에 파악할 수 있게 한다.

Panel 1: STI profile across layers (attention + MLP)
Panel 2: Effective rank comparison (IF vs Math, colored by layer depth)
Panel 3: Mean cross-cosine similarity by layer
Panel 4: 99% energy rank ratio (% of min_dim needed)
"""

fig = plt.figure(figsize=(26, 18))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)
fig.suptitle(
    "SVD Task Interference Analysis Summary\n"
    f"Base: {BASE_MODEL_ID} | Tasks: IF vs Math | {len(svd_results)} parameters",
    fontsize=18, fontweight="bold", y=0.98,
)

# ── Panel 1: STI Profile ──
ax1 = fig.add_subplot(gs[0, 0])
ax1.fill_between(
    layer_df["layer_idx"], layer_df["sti_attn"],
    alpha=0.3, color="steelblue", label="Attention",
)
ax1.fill_between(
    layer_df["layer_idx"], layer_df["sti_mlp"],
    alpha=0.3, color="indianred", label="MLP",
)
ax1.plot(layer_df["layer_idx"], layer_df["sti"], "D-", color="dimgray",
         linewidth=2.5, markersize=5, label="Total")
ax1.set_title("STI vs Layer Depth", fontsize=14)
ax1.set_xlabel("Layer Index")
ax1.set_ylabel("STI (sum)")
ax1.legend()
ax1.set_xlim(-0.5, 27.5)

# ── Panel 2: Effective Rank Comparison ──
ax2 = fig.add_subplot(gs[0, 1])
scatter2 = ax2.scatter(
    param_df["effective_rank_if"], param_df["effective_rank_math"],
    c=param_df["layer_idx"], cmap="viridis", s=40, alpha=0.7, edgecolors="gray",
)
max_r = max(param_df["effective_rank_if"].max(), param_df["effective_rank_math"].max())
ax2.plot([0, max_r], [0, max_r], "k--", alpha=0.4)
plt.colorbar(scatter2, ax=ax2, label="Layer")
ax2.set_title("Effective Rank: IF vs Math", fontsize=14)
ax2.set_xlabel("IF Effective Rank")
ax2.set_ylabel("Math Effective Rank")
ax2.set_aspect("equal")

# ── Panel 3: Mean Cross-Cosine by Layer ──
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(
    layer_df["layer_idx"], layer_df["mean_cross_cos_left"],
    "o-", color="steelblue", linewidth=2, label="Left (U)",
)
ax3.plot(
    layer_df["layer_idx"], layer_df["mean_cross_cos_right"],
    "s-", color="indianred", linewidth=2, label="Right (V)",
)
ax3.set_title("Mean |Cross-Cosine| by Layer", fontsize=14)
ax3.set_xlabel("Layer Index")
ax3.set_ylabel("Mean |Cosine Similarity|")
ax3.legend()
ax3.set_xlim(-0.5, 27.5)

# ── Panel 4: 99% Rank Ratio ──
ax4 = fig.add_subplot(gs[1, 1])
# Layer별 평균 rank_99_ratio
ratio_if = param_df.groupby("layer_idx")["rank_99_ratio_if"].mean()
ratio_math = param_df.groupby("layer_idx")["rank_99_ratio_math"].mean()
ax4.plot(ratio_if.index, ratio_if.values * 100, "o-", color="steelblue",
         linewidth=2, label="IF")
ax4.plot(ratio_math.index, ratio_math.values * 100, "s-", color="indianred",
         linewidth=2, label="Math")
ax4.set_title("99% Energy Rank Ratio (% of min_dim)", fontsize=14)
ax4.set_xlabel("Layer Index")
ax4.set_ylabel("Rank / min_dim (%)")
ax4.legend()
ax4.set_xlim(-0.5, 27.5)

fig.savefig(ARTIFACT_DIR / "summary_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACT_DIR / 'summary_dashboard.png'}")

In [ ]:
"""Cell 13: Save All Artifacts.

분석 메타데이터(JSON)와 원본 singular values(NPZ)를 저장한다.
NPZ 파일은 후속 분석(DARE, low-rank merge 등)에서 재사용할 수 있다.
"""

# ── JSON 메타데이터 ──
metadata = {
    "created_at_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "base_model": BASE_MODEL_ID,
    "if_model": str(IF_MODEL_PATH),
    "math_model": str(MATH_MODEL_PATH),
    "topk_singular_vectors": TOPK_SINGULAR_VECTORS,
    "energy_thresholds": ENERGY_THRESHOLDS,
    "include_embedding": INCLUDE_EMBEDDING,
    "svd_device": SVD_DEVICE,
    "num_parameters_analyzed": len(svd_results),
    "global_summary": {
        "mean_sti": float(param_df["sti"].mean()),
        "max_sti": float(param_df["sti"].max()),
        "max_sti_param": param_df.loc[param_df["sti"].idxmax(), "param_name"],
        "mean_effective_rank_if": float(param_df["effective_rank_if"].mean()),
        "mean_effective_rank_math": float(param_df["effective_rank_math"].mean()),
        "mean_rank_95_if": float(param_df["rank_95_if"].mean()),
        "mean_rank_95_math": float(param_df["rank_95_math"].mean()),
        "mean_rank_99_if": float(param_df["rank_99_if"].mean()),
        "mean_rank_99_math": float(param_df["rank_99_math"].mean()),
        "mean_rank_99_ratio_if": float(param_df["rank_99_ratio_if"].mean()),
        "mean_rank_99_ratio_math": float(param_df["rank_99_ratio_math"].mean()),
    },
}

metadata_path = ARTIFACT_DIR / "svd_analysis_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata: {metadata_path}")

# ── NPZ: 원본 singular values (후속 분석용) ──
sv_data = {}
for r in svd_results:
    # 키 형식: 'model.layers.14.self_attn.q_proj.weight__if'
    sv_data[f"{r.param_name}__if"] = r.singular_values_if
    sv_data[f"{r.param_name}__math"] = r.singular_values_math

npz_path = ARTIFACT_DIR / "singular_values.npz"
np.savez_compressed(str(npz_path), **sv_data)
print(f"Saved singular values: {npz_path}")

# ── 전체 artifact 목록 출력 ──
print("\n" + "=" * 60)
print("All artifacts:")
print("=" * 60)
for p in sorted(ARTIFACT_DIR.glob("*")):
    size_mb = p.stat().st_size / 1e6
    print(f"  {p.name:45s} {size_mb:.2f} MB")

## Interpretation Guide

### STI Values
| STI Range | Interpretation | Merging Strategy |
|-----------|---------------|------------------|
| ~0 | Task subspace 완전 직교 | Naive averaging 안전 |
| Low | 약한 간섭, 공유 feature 적음 | 단순 weighted average 가능 |
| High | 강한 subspace overlap | Task-specific routing, DARE, TIES, 또는 SVD-based decorrelation 필요 |

### 99% Energy Rank Ratio
| Ratio | Interpretation |
|-------|---------------|
| < 5% | 매우 low-rank → 적극적 압축 가능 |
| 5-15% | 중간 low-rank → 선택적 압축 |
| > 15% | 비교적 full-rank → 압축 시 정보 손실 주의 |

### Cross-Cosine Similarity
- **높은 mean |cos|**: 전반적으로 task 간 feature가 많이 공유됨
- **높은 max |cos|**: 특정 singular vector가 강하게 정렬됨 → 해당 방향에서 targeted intervention 필요
- **Left vs Right 비대칭**: 입력 공간(V)과 출력 공간(U)에서 간섭 양상이 다름

### RL 적용 시사점
1. **High-STI early layers**: 공통 환경 이해가 공유되는 영역 → decorrelation 대신 공유 보존 고려
2. **Low-STI deep layers**: Task-specific → 독립적 병합 가능
3. **비대칭 rank**: 한 task의 rank가 훨씬 낮으면 해당 task의 subspace가 더 집중적 → 가중치 조정 필요
4. **Top singular vectors의 높은 cosine**: 해당 방향은 보존하고 나머지만 병합하는 변형 전략 가능

In [ ]:
"""Cell 14: Cleanup.

분석 결과를 메모리에서 해제하고 GPU 캐시를 정리한다.
"""

del svd_results, result_map
gc.collect()
torch.cuda.empty_cache()

print("Notebook complete.")
print(f"All artifacts saved to: {ARTIFACT_DIR.resolve()}")